In [101]:
import os
import json
import jsonlines
from collections import defaultdict
import time

In [102]:
all_start = time.time()

In [103]:
# CMeIE
# A. c→(h)
out_name = 'c_to_hr_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to extract the possible aspect terms from the given text.
The output format of the task is: (Aspect Term) 
Given text: "{text}"
'''

oup_prompt = '{answer_text}'

In [104]:
data_file_list = [
    'processed_train.jsonl',
    'processed_dev.jsonl',
    'processed_test.jsonl',
]
out_file_list = [
    'train.json',
    'dev.json',
    'test.json'
]

In [105]:
read_dir = './ori/14lap/'
out_dir = './14lap/双向拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [106]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            inp = inp_prompt.format(text=data['text'])
            spo_list = [(item['aspect_term'],item['opinion_term'],item['sentiment_type']) for item in data['spo_list']]
            processed_spo_list = []
            for spo_item in spo_list:
                if spo_item not in processed_spo_list:
                    processed_spo_list.append(spo_item)
            oup = '\n'.join(['({})'.format(item[0].strip()) for item in processed_spo_list])
            oup = '```\n' + oup.strip() + '\n```'
            out_data = {
                'instruction':inp,
                'input':'',
                'output':oup,
                'text':data['text'],
                'spo_list':data['spo_list']
            }
            fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:c_to_hr_withtype
cost:0.05秒


In [107]:
# CMeIE
# B. c→(t)
out_name = 'c_to_tr_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to extract the possible opinion terms from the given text.
The output format of the task is: (Opinion Term) 
Given text: "{text}"
'''

oup_prompt = '{answer_text}'

In [108]:
read_dir = './ori/14lap/'
out_dir = './14lap/双向拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [109]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            inp = inp_prompt.format(text=data['text'])
            spo_list = [(item['aspect_term'],item['opinion_term'],item['sentiment_type']) for item in data['spo_list']]
            processed_spo_list = []
            processed_spo_list = []
            for spo_item in spo_list:
                if spo_item not in processed_spo_list:
                    processed_spo_list.append(spo_item)
            oup = '\n'.join(['({})'.format(item[1].strip()) for item in processed_spo_list])
            oup = '```\n' + oup.strip() + '\n```'
            out_data = {
                'instruction':inp,
                'input':'',
                'output':oup,
                'text':data['text'],
                'spo_list':data['spo_list']
            }
            fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:c_to_tr_withtype
cost:0.08秒


In [110]:
# CMeIE
# C. h[s1]c→(t,r)
out_name = 'sc_to_tr_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to identify opinion term from the given text and the aspect term. Then, determine the sentiment type.
Given the list of sentiment types: ['NEG', 'NEU', 'POS']
The output format of the task is: (Opinion Term||Sentiment Type) 
Given text: "{text}"
Given aspect term: "{aspect_term}"
'''
oup_prompt = '{answer_text}'

In [111]:
read_dir = './ori/14lap/'
out_dir = './14lap/双向拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [112]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            spo_def_list = defaultdict(list)
            for spo_item in data['spo_list']:
                aspect_term = spo_item['aspect_term']
                opinion_term = spo_item['opinion_term']
                sentiment_type = spo_item['sentiment_type']
                spo_def_list[aspect_term].append((opinion_term, sentiment_type))
            for sub in spo_def_list.keys():
                inp = inp_prompt.format(text=data['text'],aspect_term=sub)
                spo_list = spo_def_list[sub]
                oup = '\n'.join(['({}||{})'.format(item[0].strip(),item[1].strip()) for item in spo_list])
                oup = '```\n' + oup.strip() + '\n```'
                out_data = {
                    'instruction':inp,
                    'input':'',
                    'output':oup,
                    'text':data['text'],
                    'spo_list':data['spo_list']
                }
                fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:sc_to_tr_withtype
cost:0.06秒


In [113]:
# CMeIE
# D. t[s1]c→(h,r). 
out_name = 'sc_to_hr_withtype'
inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to identify aspect term from the given text and the opinion term. Then, determine the sentiment type.
Given the list of sentiment types: ['NEG', 'NEU', 'POS']
The output format of the task is: (Aspect Term||Sentiment Type) 
Given text: "{text}"
Given opinion term: "{opinion_term}"
'''
oup_prompt = '{answer_text}'

In [114]:
read_dir = './ori/14lap/'
out_dir = './14lap/双向拆解'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [115]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            spo_def_list = defaultdict(list)
            for spo_item in data['spo_list']:
                aspect_term = spo_item['aspect_term']
                opinion_term = spo_item['opinion_term']
                sentiment_type = spo_item['sentiment_type']
                spo_def_list[opinion_term].append((aspect_term, sentiment_type))
            for obj in spo_def_list.keys():
                inp = inp_prompt.format(text=data['text'],opinion_term=obj)
                spo_list = spo_def_list[obj]
                oup = '\n'.join(['({}||{})'.format(item[0].strip(),item[1].strip()) for item in spo_list])
                oup = '```\n' + oup.strip() + '\n```'
                out_data = {
                    'instruction':inp,
                    'input':'',
                    'output':oup,
                    'text':data['text'],
                    'spo_list':data['spo_list']
                }
                fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:sc_to_hr_withtype
cost:0.06秒


In [116]:
# CMeIE
# A0. c→(h,r,t)
out_name = 'c_to_hrt_withtype'

inp_prompt = '''You are currently a senior expert in fine-grained sentiment extraction.
Your task is to aspect sentiment triplets from the given text, each triplet contains aspect term, opinion term and sentiment type.
Given the list of sentiment types: ['NEG', 'NEU', 'POS']
The output format of the task is: (Aspect Term||Opinion Term||Sentiment Type) 
Given text: "{text}"
'''


oup_prompt = '{answer_text}'

In [117]:
read_dir = './ori/14lap/'
out_dir = './14lap'
out_dir = os.path.join(out_dir,out_name)
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [118]:
start = time.time()
print('out_name:{}'.format(out_name))
for data_file,out_file in zip(data_file_list,out_file_list):
    read_path = os.path.join(read_dir,data_file)
    out_path = os.path.join(out_dir,out_file)
    with jsonlines.open(read_path,'r') as f:
        datas = [data for data in f]
    with jsonlines.open(out_path,'w') as fw:
        for data in datas:
            inp = inp_prompt.format(text=data['text'])
            spo_list = [(item['aspect_term'],item['opinion_term'],item['sentiment_type']) for item in data['spo_list']]
            processed_spo_list = []
            for spo_item in spo_list:
                if spo_item not in processed_spo_list:
                    processed_spo_list.append(spo_item)
            oup = '\n'.join(['({}||{}||{})'.format(item[0].strip(),item[1].strip(),item[2].strip()) for item in processed_spo_list])
            oup = '```\n' + oup.strip() + '\n```'
            out_data = {
                'instruction':inp,
                'input':'',
                'output':oup,
                'text':data['text'],
                'spo_list':data['spo_list']
            }
            fw.write(out_data)
end = time.time()
print('cost:{}秒'.format(round(end-start, 2)))

out_name:c_to_hrt_withtype
cost:0.04秒


In [119]:
all_end = time.time()
all_end - all_start

0.48029065132141113